In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

# =============================================================================
# STAGE 1 & 2: DATA SETUP
# =============================================================================
print("--- STAGE 1 & 2: SETTING UP DATA ---")

# --- IMPORTANT: Ensure this path is correct! ---
# dataset_path = './Garbage classification/Garbage classification' 
dataset_path = './garbage_classification'

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

# Create datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path, validation_split=0.2, subset="training", seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path, validation_split=0.2, subset="validation", seed=123,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"\nFound {num_classes} class names: {class_names}")

# Create test set
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take(val_batches // 2)
val_ds = val_ds.skip(val_batches // 2)

# Optimize datasets
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# =============================================================================
# STAGE 3: MODEL BUILDING AND TRAINING WITH CALLBACKS
# =============================================================================
print("\n--- STAGE 3: BUILDING AND TRAINING THE BEST MODEL ---")

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

model = keras.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    data_augmentation,
    layers.Rescaling(1./255),
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Flatten(),
    layers.Dense(128),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# --- THE KEY CHANGE IS HERE ---
# We are using the default 'adam' optimizer which has a learning rate of 0.001
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# --- Define Callbacks ---
# We'll increase patience slightly to be safe
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, verbose=1, restore_best_weights=True)
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_waste_classifier.keras', monitor='val_loss', save_best_only=True, verbose=1)

epochs = 25
print(f"\nStarting model training for up to {epochs} epochs...")
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs,
  callbacks=[early_stopping, model_checkpoint]
)
print("\nModel training finished.")

# =============================================================================
# STAGE 4: LOAD AND EVALUATE THE BEST MODEL
# =============================================================================
print("\n--- STAGE 4: LOADING AND EVALUATING THE BEST SAVED MODEL ---")

# Because we used restore_best_weights=True in EarlyStopping, the 'model' object
# in memory is already the best version. We can evaluate it directly.
print("\nEvaluating the best model on the unseen test set...")
final_loss, final_accuracy = model.evaluate(test_ds)
print(f'\nFinal Test Loss of Best Model: {final_loss:.4f}')
print(f'Final Test Accuracy of Best Model: {final_accuracy*100:.2f}%')

--- STAGE 1 & 2: SETTING UP DATA ---
Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.

Found 6 class names: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

--- STAGE 3: BUILDING AND TRAINING THE BEST MODEL ---

Starting model training for up to 25 epochs...
Epoch 1/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4032 - loss: 1.6700
Epoch 1: val_loss improved from inf to 1.98772, saving model to best_waste_classifier.keras
64/64 ━━━━━━━━━━━━━━━━━━━━ 235s 4s/step - accuracy: 0.4040 - loss: 1.6674 - val_accuracy: 0.2369 - val_loss: 1.9877
Epoch 2/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5369 - loss: 1.2583
Epoch 2: val_loss did not improve from 1.98772
64/64 ━━━━━━━━━━━━━━━━━━━━ 212s 3s/step - accuracy: 0.5372 - loss: 1.2575 - val_accuracy: 0.2570 - val_loss: 2.2534
Epoch 3/25
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5910 - loss: 1.1064
Epoch 3: val